# c2-build1-brief-C-group-10 · Brief C · Group 10

**What this notebook produces:** The H1 2026 invoice and payment reconciliation bridge, saved to `outputs/brief_c/finance_bridge_summary.csv` and `.json`.

**What it needs:** Run `clean_brief_c.py` and `analyze_brief_c_reconciliation.py` on the local data pack first; this preview reads only the committed aggregate bridge.

**Owner:** Rishabh. **Approach-log rows:** 6, 16–18.

Every section follows the same five cells: MAP (where this sits and what this section adds), DO (one idea per code cell), SEE (print the output), CHECK (an assert that prints pass or fail without stopping the class), SUM (what we now know, with its denominator).

## CONFIG

The project label and location of the aggregate Brief C output are set here.

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG = {"brief": "Brief C", "group": 10, "output_dir": ROOT / "outputs" / "brief_c"}
import pandas as pd


## 1 · MAP

>This notebook sits at the end of the pipeline: it brings together the raw source files, profiling, cleaning, and reconciliation, then presents the H1 2026 finance answer. It records the steps, checks, and outputs so the result can be reviewed and reproduced.

This project first established 4.97% H1 growth by scheduled test date (rounds to 5%), cleaned and explored the data, and used Brief A/B as supporting analysis. Group 10 selected Brief C for the COO's board-facing question: reconcile H1 invoices, captured payments, late captures, and unmatched attempts.


In [ ]:
raw = pd.read_csv(CONFIG["output_dir"] / "finance_bridge_summary.csv")
print(raw.shape)

The committed aggregate bridge is the short executive view; exact amounts and matching rules are in the JSON audit summary.

In [ ]:
raw.head()

Show the committed bridge rows, which include row counts, crore amounts, and exact rupees.


In [ ]:
def check(name, condition, detail=''):
    print(('PASS ' if condition else 'FAIL ') + name + ('  ' + str(detail) if detail != '' else ''))

# example checks: replace with yours
check('rows present', len(raw) > 0, len(raw))
# after a cleaning step:
# check('input = clean + rejected', len(raw) == len(clean) + len(rejected), (len(raw), len(clean), len(rejected)))
# after a join:
# check('join did not fan out', joined.shape[0] == left.shape[0], (left.shape[0], joined.shape[0]))

### SUM

The committed H1 bridge reports ₹1.947 crore in positive balances across 11,586 invoices at 30 June; after later captures, ₹1.648 crore remained open across 9,824 invoices at 15 July. The 1,541 unmatched H1 captures worth ₹0.267 crore are a separate Finance exception, not a confirmed loss.

## 2 · What Brief C tells Finance


## Break it on purpose: a join that doubles billed value

One invoice can have several payment attempts. Joining invoice rows directly to attempts repeats the invoice amount; aggregate captures at invoice grain before calculating the balance.

In [ ]:
demo_invoices = pd.DataFrame({"invoice_id": ["INV-1"], "amount_rs": [100]})
demo_attempts = pd.DataFrame({"invoice_id": ["INV-1", "INV-1"], "captured_rs": [60, 50]})
unsafe = demo_invoices.merge(demo_attempts, on="invoice_id", how="left")
safe = demo_attempts.groupby("invoice_id", as_index=False).agg(captured_rs=("captured_rs", "sum")).merge(
    demo_invoices, on="invoice_id", how="right", validate="one_to_one")
print("billed before:", demo_invoices.amount_rs.sum(), "after unsafe join:", unsafe.amount_rs.sum())
print("billed after safe aggregation:", safe.amount_rs.sum(), "captured:", safe.captured_rs.sum())

## Output

This executive preview reads the committed aggregate bridge only; it does not load private row-level finance data. The detailed JSON carries the matching rule, exact amounts, timing and limitations.

In [ ]:
print("Brief C bridge:", CONFIG["output_dir"] / "finance_bridge_summary.csv")
print(f"{len(raw)} audit rows · H1 2026 · Group 10")

## Quick project answers · Brief C · Group 10

- **Why is growth 5%, not 18%?** Active H1 scheduled test lines rose from 435,880 to 457,539: +4.97%, which rounds to 5%; 18.32% is the plan's implied growth, not actual growth.
- **Why Brief C?** The COO's invoice-to-capture question can be answered with explicit periods, amounts, row counts, safe matching rules, and Finance action lists.
- **What supports the board answer?** H1 invoice rows, captured status, 30 June close, 15 July observation cutoff, exact rupee summaries, and separately reported unmatched captures.